In [1]:
import Python_Modules.HelperFunctions as HelperFunctions
from transformers import AutoModelForImageClassification, AutoConfig, AutoModelForCausalLM
import os
os.chdir('/media/my_drives')
import pandas as pd
from tqdm import tqdm
import pickle
import torch
import numpy as np
import json
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

/media/my_drives/DATA4/environments/max/ImageBenchmark/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-27 12:09:02.538496: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761566942.548941 2618428 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761566942.552099 2618428 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1761566942.560423 2618428 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the s

In [2]:
datasets = [
 'AIDA',
 'Brand-Styles',
 'BrandSelfie',
 'E-Commerce',
 'Emotion-6',
 'FILGRIM',
 'FashionImages',
 'GenImgNet',
 'Generated-Ethnicity',
 'Generated-Gender',
 'KonIQ',
 'LogoDet-3K',
 'StoreItemColor',
 'UnBiasedEmo',
 'Unsplash-Images',
 'good-bad',
 'intel-sceneries',
 'sentiment']


In [9]:
DF_BASE_PATH = 'DATA4/max/Image_Benchmark/R1_Analytics/CSV_Holdout'
TRAIN_SIZE = '200'
PRED_COL = 'prediction_convnext_allclassprobs_200' #_200

In [10]:
for DATASET in tqdm(datasets):
    df = pd.read_csv(f'{DF_BASE_PATH}/{DATASET}.csv')
    #id2label = dict(zip(df['label_num'].astype(int), df['label']))

    model = AutoModelForImageClassification.from_pretrained(
    f'DATA4/max/Image_Benchmark/R1_Analytics/Finetuned_Convnext_Models_ForEmbeds/{DATASET}_{TRAIN_SIZE}'
).to(device)
    processor = AutoImageProcessor.from_pretrained(
        f'DATA4/max/Image_Benchmark/R1_Analytics/Finetuned_Convnext_Models_ForEmbeds/{DATASET}_{TRAIN_SIZE}'
    )

    #if PRED_COL not in df.columns:
    df[PRED_COL] = np.nan

    for i in df.loc[df[PRED_COL].isna()].index:
        image_path = df.at[i, 'image_path']
        image = Image.open(image_path).convert('RGB')

        # Preprocess
        inputs = processor(images=image, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Forward pass
        with torch.no_grad():
            outputs = model(**inputs)
        
        # logits → probabilities
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        
        # Optional: convert to dictionary with class labels
        class_probs = {model.config.id2label[i]: np.round(float(probs[0, i]), 6) for i in range(probs.shape[1])}
        df.at[i, PRED_COL] = json.dumps(class_probs)
        
        if i % 200 == 0:
            df.to_csv(f'{DF_BASE_PATH}/{DATASET}.csv', index=False)
        df.to_csv(f'{DF_BASE_PATH}/{DATASET}.csv', index=False)

  0%|                                                     | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_2618428/3351296675.py:33: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '{"0.0": 0.000175, "1.0": 9.8e-05, "10.0": 4.6e-05, "11.0": 0.0002, "14.0": 0.00028, "15.0": 0.000218, "16.0": 0.000305, "17.0": 0.000117, "19.0": 0.000342, "2.0": 0.000695, "21.0": 0.000235, "22.0": 0.0001, "23.0": 0.0002, "25.0": 0.000166, "27.0": 0.000117, "29.0": 0.000212, "30.0": 0.00091, "31.0": 0.000105, "32.0": 0.994394, "4.0": 0.000306, "5.0": 0.000214, "7.0": 0.000374, "9.0": 0.000126, "nan": 6.7e-05}' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[i, PRED_COL] = json.dumps(class_probs)
100%|█████████████████████████████████████████████| 1/1 [00:12<00:00, 12.98s/it]
